# Image FetchPush conedir — OFFLINE ORIGINAL-STYLE qualification (fixed alpha=0 + BC 0.05 + twin critics)

Strictly-offline (seed 0) qualification of the *original-style* actor on pixels:

```
actor_loss = 0.05 * BC_NLL(data_action | state, future_goal)
           + 0.95 * (-min_Q(state, policy_action, future_goal))
```

`entropy_coefficient=0.0` (fixed alpha=0, no adaptive optimizer) · `bc_coef=0.05` ·
`twin_q=True` (actor uses min(Q1,Q2)) · `random_goals=0.0` (same-trajectory positives) ·
frozen `.npz` IS the buffer (zero env interaction). All eval distances are **physical
object-goal coordinates** (sim), never flattened image-L2.

Pipeline: pinned install → clone+checkout → mount Drive → env/render audit →
**collect-or-reuse the frozen dataset** → `scripts/qualify_image_offline_original_style`
(preflight → 100k offline → physical fixed-eval → checkpoint diagnostics → GIFs → verdict).

> ⚠️ Requires commit **a6d8373** (or later): the driver + physical-eval instrumentation
> + the final-save None-guard fix. Set `COMMIT` in cell 2 accordingly.

In [1]:
# 1. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX (proven-safe recipe).
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"                         # headless render backend on Colab
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}", f"numpy=={numpy.__version__}"]
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps","dm-haiku","optax","chex")
pip("jmp","tabulate","toolz","etils","tensorboardX","gymnasium-robotics","mujoco","imageio-ffmpeg",*hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())

Colab JAX 0.7.2 | devices: [CudaDevice(id=0)]
post-install JAX 0.7.2 | devices: [CudaDevice(id=0)]


In [2]:
# 2. Clone the fork and checkout the commit that CONTAINS the instrumentation + driver + fix.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
COMMIT = "a6d8373"   # driver + physical-eval instrumentation + final-save None-guard fix
!git fetch -q origin && git checkout -q $COMMIT
!git log -1 --oneline
assert os.path.exists("scripts/qualify_image_offline_original_style.py"),     "checkout is MISSING the qualification driver -- set COMMIT to a6d8373 or later."
assert os.path.exists("scripts/collect_push_dataset.py"), "missing dataset collector."

Cloning into 'contrastive_rl'...
remote: Enumerating objects: 676, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 676 (delta 127), reused 142 (delta 55), pack-reused 444 (from 1)
Receiving objects: 100% (676/676), 23.90 MiB | 16.16 MiB/s, done.
Resolving deltas: 100% (336/336), done.
/content/contrastive_rl
a6d8373 (HEAD) checkpoint: fix TypeError when save_checkpoint gets success=None (final save)


In [3]:
# 3. Mount Google Drive (checkpoints + artifacts saved here).
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# 4. Paths. Frozen dataset is COLLECTED ONCE (cell 6), then reused. New run dir.
import os, json, shutil, numpy as np
ENV  = "fetch_push_image_conedir"
SEED = 0
SMOKE = True     # True = tiny integration smoke; set False for the real 100k qualification

BASE = "/content/drive/MyDrive/easypush_image_qual"
DATA = f"{BASE}/data/push_image_conedir_noisy_oracle_s{SEED}" + ("_smoke" if SMOKE else "") + ".npz"
RUN_DIR = f"{BASE}/fetch_push_image_conedir_off_bc005_alpha0_twin_s{SEED}" + ("_smoke" if SMOKE else "")
OUT = "artifacts/image_conedir_offline_original_style"
STEPS = 2000 if SMOKE else 100000
N_EPISODES = 50 if SMOKE else 1000
_finished = os.path.exists(os.path.join(RUN_DIR, "final.pkl"))
if SMOKE and _finished:
    shutil.rmtree(RUN_DIR)                      # smoke is disposable -> start clean
    print("cleared previous smoke run dir:", RUN_DIR)
elif _finished:
    raise SystemExit(f"{RUN_DIR} already has a FINISHED real run; pick a fresh dir or "
                     f"pass --resume to continue rather than overwrite.")
print("DATA   :", DATA, "(exists)" if os.path.exists(DATA) else "(will collect)")
print("RUN_DIR:", RUN_DIR)

cleared previous smoke run dir: /content/drive/MyDrive/easypush_image_qual/fetch_push_image_conedir_off_bc005_alpha0_twin_s0_smoke
DATA   : /content/drive/MyDrive/easypush_image_qual/data/push_image_conedir_noisy_oracle_s0_smoke.npz (exists)
RUN_DIR: /content/drive/MyDrive/easypush_image_qual/fetch_push_image_conedir_off_bc005_alpha0_twin_s0_smoke


In [5]:
# 5. Environment/render audit -- STOP before anything if it fails.
!python -m scripts.smoke_image_conedir
audit = json.load(open("artifacts/push_image_probe/smoke_image_conedir.json"))
assert audit["verdict"] == "PASS", "image-conedir env audit FAILED -- stopping."
print("env audit PASS (oracle=%.2f random=%.2f)" % (
    audit["gate4_oracle"]["oracle_success"], audit["gate4_oracle"]["random_success"]))

AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details
gate1 WIRING      PASS dtype=uint8 shape=(24576,) frame_std=60.0
gate2 NO_LEAK     PASS leak_pixels=0 target0_alpha=0.0
gate3 GOAL_IMAGE  PASS pixel_diff=['10.98', '15.96', '17.59'] goal_match=['0.0e+00', '0.0e+00', '0.0e+00']
gate4 ORACLE      PASS oracle=1.00 random=0.60
gate5 TRAIN_STEP  PASS critic=0.691 actor=0.016

VERDICT: PASS  (artifacts in artifacts/push_image_probe)
env audit PASS (oracle=1.00 random=0.60)


In [6]:
# 6. Collect the FROZEN dataset (ONCE; reused if it already exists). NOT a download --
#    it renders the FetchPush image env with a noisy scripted oracle + a random-episode
#    fraction. ~15s for 50 smoke episodes / ~5 min for 1000 on a T4.
os.makedirs(os.path.dirname(DATA), exist_ok=True)
if os.path.exists(DATA):
    meta = json.loads(str(np.load(DATA, allow_pickle=True)["meta"]))
    print("reusing existing dataset:", DATA)
else:
    from scripts.collect_push_dataset import collect
    print(f"collecting {N_EPISODES} episodes -> {DATA}")
    meta = collect(ENV, episodes=N_EPISODES, noise=0.3, random_frac=0.2, seed=SEED, out=DATA)
print(json.dumps(meta, indent=2))
assert meta["behavior_success_oracle"] and meta["behavior_success_oracle"] >= 0.9,     "demonstrator broken -- oracle arm should succeed >=0.9"
assert os.path.exists(DATA), "dataset collection failed"
print("dataset ready, behavior_success =", meta["behavior_success"])

reusing existing dataset: /content/drive/MyDrive/easypush_image_qual/data/push_image_conedir_noisy_oracle_s0_smoke.npz
{
  "env_name": "fetch_push_image_conedir",
  "episodes": 50,
  "seed": 0,
  "noise": 0.3,
  "random_frac": 0.2,
  "obs_dim": 12288,
  "goal_dim": 12288,
  "action_dim": 4,
  "start_index": 0,
  "end_index": -1,
  "goal_indices": null,
  "use_image_obs": true,
  "max_episode_steps": 50,
  "behavior_success": 0.88,
  "behavior_success_oracle": 0.975,
  "behavior_success_random": 0.5
}
dataset ready, behavior_success = 0.88


In [7]:
# 7. Run the qualification driver: preflight -> 100k offline -> physical fixed-eval ->
#    checkpoint diagnostics (20k/50k/100k) -> rollout GIFs -> verdict. Physical metrics only.
#    On error the driver's real traceback is printed below (no swallowing).
cmd = ["python","-m","scripts.qualify_image_offline_original_style",
       "--dataset", DATA, "--run_dir", RUN_DIR, "--out", OUT,
       "--seed", str(SEED), "--steps", str(STEPS)]
if SMOKE: cmd.append("--smoke")
print(" ".join(cmd))
import subprocess
r = subprocess.run(cmd)
if r.returncode != 0:
    raise SystemExit(f"driver failed (exit {r.returncode}); see traceback above")

python -m scripts.qualify_image_offline_original_style --dataset /content/drive/MyDrive/easypush_image_qual/data/push_image_conedir_noisy_oracle_s0_smoke.npz --run_dir /content/drive/MyDrive/easypush_image_qual/fetch_push_image_conedir_off_bc005_alpha0_twin_s0_smoke --out artifacts/image_conedir_offline_original_style --seed 0 --steps 2000 --smoke


In [9]:
import json, os
from IPython.display import Image, display
print("=== PREFLIGHT ==="); print(json.dumps(json.load(open(f"{OUT}/preflight_audit.json"))["checks"], indent=2))
summ = json.load(open(f"{OUT}/summary.json"))
print("\n=== VERDICT:", summ["qualification_verdict"], "===")
print(json.dumps(summ["final_checkpoint_physical"], indent=2))
print("\n=== fixed_eval.csv ==="); print(open(f"{OUT}/fixed_eval.csv").read())
print("=== checkpoint_diagnostics.csv ==="); print(open(f"{OUT}/checkpoint_diagnostics.csv").read())
for s in ("20000","50000","100000"):
    g = f"{RUN_DIR}/rollout_{s}.gif"
    if os.path.exists(g): display(Image(g, width=256))

=== PREFLIGHT ===
{
  "1_dataset_exists": true,
  "6_action_aligned": true,
  "10_random_goals_0": true,
  "11_twin_q_true": true,
  "12_bc_coef_0.05": true,
  "13_fixed_alpha0": true,
  "13_no_adaptive_optimizer": true,
  "14_physical_eval_on": true,
  "4_obs_uint8_image": true,
  "5_action_in_range": true
}

=== VERDICT: CRITIC_RETRIEVAL_WITHOUT_CONTROL ===
{
  "physical_success_rate": 0.0,
  "physical_final_object_goal_distance": 0.07609746605157852,
  "physical_min_object_goal_distance": 0.07609741669148207,
  "object_displacement": 0.00010776519775390625,
  "minimum_gripper_object_distance": 0.04482710640877485,
  "contact_steps": 3.25,
  "moving_object_fraction": 0.0,
  "fall_or_invalid_rate": 0.0,
  "action_mean_per_dimension": [
    -0.061396222561597824,
    0.3555837869644165,
    -0.3311779499053955,
    0.048472337424755096
  ],
  "action_std_per_dimension": [
    0.024573728442192078,
    0.033478014171123505,
    0.06261175125837326,
    0.011722665280103683
  ],
  "actio